# CICIoT Sentinel: capture-compatible retraining

This notebook trains using the same `sentinel_ml` extractor and preprocessor used by FastAPI.
The prior CSV experiment is preserved in `ciciot23_training_legacy.ipynb`. Its model is incompatible with this contract.

Prepare independent labeled Ethernet/IPv4 PCAP recordings and assign **recording sessions** to train, validation and test before extracting windows. No label determines window size. Labels come from recording schedules, never predictions. Split captures from the same experiment/device/time group together. Use representative benign load and separate single-source/multi-source attack experiments. Mixed-label recordings must first be segmented using independently recorded scenario boundaries, with all resulting segments in one capture_group.

The original pre-extracted Kaggle CSVs cannot be used in this new experiment: their measurement semantics and class-dependent windows differ.


In [ ]:
# Run in the repository checkout (on Kaggle, upload/clone the repository first).
%pip install -e . xgboost==3.4.1


## Prepare capture_manifest.json
Paths are relative to the manifest. Every split must contain all three classes and use independent capture_group values. A capture file cannot occur twice. Example entry:
`{"path":"captures/benign-session-01.pcap","capture_group":"session-01","split":"train","label":"Benign"}`.
Use real captures; no automatic data download, synthetic replacement, random row split, test-label cleaning, or imputation is performed.


In [ ]:
from pathlib import Path
from sentinel_ml.contract import SPEC, CONTRACT_HASH
from sentinel_ml.training import train
import json
MANIFEST = Path("ml/training/capture_manifest.json")
OUTPUT = Path("ml/artifacts/retrained")
print(json.dumps(SPEC, indent=2))
print("Contract hash:", CONTRACT_HASH)
assert MANIFEST.is_file(), "Prepare the labeled capture manifest described above"


## Train and evaluate
XGBoost, Random Forest and scaled Logistic Regression use the same validated 22-column float32 matrix. Scaling is fitted on training rows only. Model selection uses validation macro-F1. Only the selected model is evaluated on the final test split, once. Do not tune using that result. No test-dependent deletion is performed; exact feature overlap is reported for investigation. All model bundles carry contract hashes, source hashes, class order and golden predictions. The estimator set/hyperparameters are fixed in `sentinel_ml/training.py`; select changes using training/validation experiments before final testing.


In [ ]:
report = train(MANIFEST, OUTPUT)
print(json.dumps(report, indent=2))


## Verify the deployment artifact
Every API load repeats these golden-row checks. Copy only the new bundles to the application's configured MODEL_DIR. The old artifact remains a historical result. This run establishes capture-level offline results; live robustness, calibration, false alerts per benign hour and end-to-end latency need independent lab measurements.


In [ ]:
from sentinel_ml.artifacts import load
for model_id in report["artifact_hashes"]:
    bundle = load(OUTPUT / f"ciciot_22f_{model_id}.joblib")
    print(model_id, bundle["artifact_sha256"], "golden predictions verified")
